# Week 3: My First Neural Network
## MNIST Handwritten Digit Classification with PyTorch

**Problem:** classify a 28 × 28 grayscale image as a digit from 0 to 9.

**Objectives:** prepare image tensors; build a small neural network; train with backpropagation; plot loss and accuracy over epochs; inspect the training/validation gap; evaluate once on an untouched test set.

The notebook uses CPU execution, fixed seeds, and a small multilayer perceptron (MLP). No GPU is required. All metrics below come from actual execution.


## 1. Libraries and reproducibility

In Colab, the packages are normally preinstalled. Locally, install `torch numpy pandas matplotlib scikit-learn` and Jupyter. Fixed seeds improve repeatability; results may still differ between library versions and hardware.


In [ ]:
import copy
import hashlib
import json
import platform
import random
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
from IPython.display import display, Markdown

get_ipython().run_line_magic('matplotlib', 'inline')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
device = torch.device('cpu')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('Device:', device)


## 2. Download and inspect MNIST

MNIST contains 60,000 development images and a separate official test set of 10,000 images. Pixel values are integers from 0 to 255. This notebook downloads the same NumPy archive used by the official Keras loader, then trains the model entirely with PyTorch.

Sources: [MNIST documentation](https://keras.io/api/datasets/mnist/), [official Keras loader source](https://github.com/keras-team/keras/blob/master/keras/src/datasets/mnist.py). Dataset attribution: Yann LeCun and Corinna Cortes; see the MNIST documentation for the CC BY-SA 3.0 dataset terms.


In [ ]:
DATA_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
archive_path = data_dir / 'mnist.npz'
if not archive_path.exists():
    with urllib.request.urlopen(DATA_URL, timeout=120) as response:
        archive_path.write_bytes(response.read())

archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with np.load(archive_path, allow_pickle=False) as archive:
    x_development, y_development = archive['x_train'], archive['y_train']
    x_test_raw, y_test = archive['x_test'], archive['y_test']
assert x_development.shape == (60000, 28, 28)
assert x_test_raw.shape == (10000, 28, 28)
assert set(np.unique(y_development)) == set(range(10))
print('Development images:', x_development.shape)
print('Test images:', x_test_raw.shape)
print('Pixel range:', int(x_development.min()), 'to', int(x_development.max()))
print('Missing pixels:', int(np.isnan(x_development).sum()))
print('Archive SHA-256:', archive_sha256)
display(pd.Series(y_development).value_counts().sort_index().rename_axis('Digit').to_frame('Development examples'))


## 3. Visualize images and class balance

The ten classes are fairly balanced. A roughly 10% accuracy is a useful chance-level reference for a ten-class task, though exact class proportions differ.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    index = np.flatnonzero(y_development == digit)[0]
    ax.imshow(x_development[index], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Digit {digit}')
    ax.axis('off')
fig.suptitle('MNIST: one development example per class')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.bar(np.arange(10), np.bincount(y_development))
plt.xticks(np.arange(10))
plt.xlabel('Digit')
plt.ylabel('Development examples')
plt.title('MNIST class distribution')
plt.tight_layout()
plt.show()


## 4. Separate training, validation, and test sets

Split only the 60,000 development examples: 54,000 for training and 6,000 for validation, with stratification. Keep the official 10,000-example test set untouched until the final evaluation. Validation guides checkpoint selection; test data must not guide training decisions.

Normalize pixels to [0, 1] by dividing by 255. This fixed transformation learns no statistics from validation or test images. Images stay 28 × 28 until the model flattens them to 784 inputs.


In [ ]:
train_indices, val_indices = train_test_split(
    np.arange(len(y_development)), test_size=0.10,
    stratify=y_development, random_state=SEED
)
assert set(train_indices).isdisjoint(set(val_indices))

def make_dataset(images, labels):
    pixels = torch.from_numpy(images.astype(np.float32) / 255.0)
    targets = torch.from_numpy(labels.astype(np.int64))
    return TensorDataset(pixels, targets)

train_data = make_dataset(x_development[train_indices], y_development[train_indices])
val_data = make_dataset(x_development[val_indices], y_development[val_indices])
test_data = make_dataset(x_test_raw, y_test)
BATCH_SIZE = 256
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=0)
train_eval_loader = DataLoader(train_data, batch_size=512, shuffle=False, num_workers=0)
val_loader = DataLoader(val_data, batch_size=512, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False, num_workers=0)
print('Training:', len(train_data), '| Validation:', len(val_data), '| Test:', len(test_data))
print('Normalized training pixel range:', train_data.tensors[0].min().item(), train_data.tensors[0].max().item())


## 5. Build a small neural network

Architecture: **784 → 128 → 64 → 10**. ReLU introduces nonlinear behavior, allowing the network to learn patterns beyond a linear decision boundary. Dropout randomly disables 20% of the first hidden layer's activations during training, providing regularization.

The output consists of ten raw scores (logits). `CrossEntropyLoss` handles the required log-softmax internally, so do not add a softmax layer before this loss. Adam updates the weights using gradients calculated by backpropagation.


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Dropout(p=0.20),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print('Trainable parameters:', f'{parameter_count:,}')


## 6. Train for 15 epochs and track validation performance

One epoch is one pass through all training examples. Each batch performs a forward pass, loss calculation, gradient reset, backpropagation, and optimizer update. Validation uses `model.eval()` and disables gradients.

For a fair training/validation gap, both plotted metrics are measured in **evaluation mode after each epoch**, with dropout disabled. The training optimization loss is recorded separately. We retain the checkpoint with the lowest validation loss, without using the test set.


In [ ]:
def evaluate(loader, collect_predictions=False):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    probabilities, labels = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            total_loss += criterion(logits, targets).item() * len(targets)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total += len(targets)
            if collect_predictions:
                probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
                labels.append(targets.cpu().numpy())
    metrics = {'loss': total_loss / total, 'accuracy': total_correct / total}
    if collect_predictions:
        return metrics, np.concatenate(probabilities), np.concatenate(labels)
    return metrics

EPOCHS = 15
history_rows = []
best_val_loss = float('inf')
best_state = None
best_epoch = None
started = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimization_loss, total = 0.0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        optimization_loss += loss.item() * len(targets)
        total += len(targets)
    training = evaluate(train_eval_loader)
    validation = evaluate(val_loader)
    history_rows.append({
        'epoch': epoch, 'optimization_loss': optimization_loss / total,
        'train_loss': training['loss'], 'val_loss': validation['loss'],
        'train_accuracy': training['accuracy'], 'val_accuracy': validation['accuracy']
    })
    if validation['loss'] < best_val_loss:
        best_val_loss = validation['loss']
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
    print(f"Epoch {epoch:02d}/{EPOCHS}: train loss={training['loss']:.4f}, val loss={validation['loss']:.4f}, "
          f"train accuracy={training['accuracy']:.2%}, val accuracy={validation['accuracy']:.2%}")
elapsed_seconds = time.perf_counter() - started
history = pd.DataFrame(history_rows)
print(f'Elapsed training/evaluation time: {elapsed_seconds:.1f} seconds')
print('Best epoch by validation loss:', best_epoch)
display(history.round(4))


## 7. Plot loss and accuracy over epochs

A widening training/validation accuracy gap, or validation loss rising while training loss falls, can indicate overfitting. A small gap alone does not prove good generalization. The dashed line marks the selected validation checkpoint. Curves below are computed from this run.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.epoch, history.train_loss, marker='o', label='Train (eval mode)')
axes[0].plot(history.epoch, history.val_loss, marker='o', label='Validation')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Loss over epochs')
axes[1].plot(history.epoch, history.train_accuracy, marker='o', label='Train (eval mode)')
axes[1].plot(history.epoch, history.val_accuracy, marker='o', label='Validation')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy over epochs')
for ax in axes:
    ax.axvline(best_epoch, color='gray', linestyle='--', label=f'Selected epoch {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_xticks(range(1, EPOCHS + 1, 2))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. Evaluate the selected checkpoint on the untouched test set

Restore the weights selected by validation loss, then evaluate the test set once. Accuracy measures overall correct classifications. Macro precision, recall, and F1 average each digit's score equally. The classification report also shows performance for individual digits.


In [ ]:
assert best_state is not None
model.load_state_dict(best_state)
test_metrics, test_probabilities, actual = evaluate(test_loader, collect_predictions=True)
predicted = test_probabilities.argmax(axis=1)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(actual, predicted, average='macro', zero_division=0)
assert np.isclose(test_metrics['accuracy'], accuracy_score(actual, predicted))
test_summary = pd.DataFrame([{
    'Test loss': test_metrics['loss'], 'Test accuracy': test_metrics['accuracy'],
    'Macro precision': macro_precision, 'Macro recall': macro_recall, 'Macro F1': macro_f1
}], index=['Selected MLP'])
display(test_summary.round(4))
print(classification_report(actual, predicted, digits=4, zero_division=0))


## 9. Confusion matrix and sample predictions

Rows show actual digits and columns show predicted digits. Off-diagonal counts reveal mistakes. The sample grid uses a fixed random seed; it was not selected to show only correct predictions. Color indicates whether the predicted label is correct.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(actual, predicted, labels=list(range(10)))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('MNIST test confusion matrix: selected checkpoint')
plt.tight_layout()
plt.show()

rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(actual), size=16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for index, ax in zip(sample_indices, axes.ravel()):
    correct = actual[index] == predicted[index]
    ax.imshow(x_test_raw[index], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'True {actual[index]} | Pred {predicted[index]}', color='green' if correct else 'red', fontsize=9)
    ax.axis('off')
fig.suptitle('Fixed random sample of test predictions')
plt.tight_layout()
plt.show()


## 10. Inspect mistakes

Inspecting errors can reveal ambiguous handwriting and visually similar digits. Even a confident prediction can be wrong; softmax scores are not guaranteed to be calibrated probabilities.


In [ ]:
mistakes = np.flatnonzero(actual != predicted)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, index in zip(axes.ravel(), mistakes[:10]):
    ax.imshow(x_test_raw[index], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'True {actual[index]} → Pred {predicted[index]}\nScore {test_probabilities[index, predicted[index]]:.2f}', fontsize=10)
for ax in axes.ravel():
    ax.axis('off')
fig.suptitle('First ten test mistakes in original test order')
plt.tight_layout()
plt.show()
print('Total misclassified test images:', len(mistakes))


## 11. Results and key learnings

The interpretation below is derived from the actual training history. The selected epoch can differ from the final epoch because more training does not always improve validation performance.


In [ ]:
selected = history.loc[history.epoch == best_epoch].iloc[0]
final = history.iloc[-1]
selected_gap = selected.train_accuracy - selected.val_accuracy
final_gap = final.train_accuracy - final.val_accuracy
late_loss_rise = final.val_loss > history.val_loss.min() + 1e-6
observations = (
    f"**Selected epoch:** {best_epoch} of {EPOCHS}, using the lowest validation loss.\n\n"
    f"**Selected checkpoint:** training accuracy {selected.train_accuracy:.2%}; validation accuracy {selected.val_accuracy:.2%}; "
    f"gap {selected_gap * 100:.2f} percentage points.\n\n"
    f"**Test results:** accuracy {test_metrics['accuracy']:.2%}; cross-entropy loss {test_metrics['loss']:.4f}; "
    f"macro F1 {macro_f1:.4f}; {len(mistakes)} mistakes out of {len(actual):,} images.\n\n"
    f"**Final epoch gap:** {final_gap * 100:.2f} percentage points. "
    + ("Final validation loss was above its best value, so keeping the best validation checkpoint avoided relying on the final weights. "
       if late_loss_rise else "The final epoch also had the lowest validation loss. ")
    + "A training/validation gap is a diagnostic signal, not a guarantee about future data.\n\n"
    "**Key learnings:** image normalization, nonlinear layers, dropout, backpropagation, batching, "
    "evaluation mode, validation-based checkpoint selection, and honest separation of training, validation, and test data."
)
display(Markdown(observations))


## 12. Limitations and next steps

- One random seed and one validation split do not quantify uncertainty.
- MNIST consists of centered grayscale digits; performance does not establish accuracy on phone photos, colored backgrounds, or other handwriting collections.
- This MLP flattens the image, so it has no explicit spatial inductive bias like a convolutional network.
- Dropout may reduce overfitting, but this experiment does not compare against a no-dropout model and cannot attribute gains to dropout alone.
- Similar examples may occur across the official splits; this project preserves the standard MNIST benchmark rather than claiming duplicate-free splits.
- Future work: compare a small CNN, inspect per-class errors, test new handwriting, and use validation data for tuning. Keep the test set reserved for final evaluation.

Reference for the training loop: [PyTorch optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).


## 13. Reproducibility check and export

This final cell verifies outputs and saves the measured epoch history and run summary into an `artifacts` folder. Generated results are not hard-coded. The notebook itself preserves plots and tables after saving.


In [ ]:
assert len(history) == EPOCHS
assert np.isfinite(history.select_dtypes(include='number').to_numpy()).all()
assert len(predicted) == 10000
assert cm.sum() == len(actual)
assert all(0 <= value <= 1 for value in [test_metrics['accuracy'], macro_precision, macro_recall, macro_f1])
artifact_dir = Path('artifacts')
artifact_dir.mkdir(exist_ok=True)
history.to_csv(artifact_dir / 'training-history.csv', index=False)
run_summary = {
    'seed': SEED, 'epochs': EPOCHS, 'best_epoch': int(best_epoch), 'parameters': parameter_count,
    'train_examples': len(train_data), 'validation_examples': len(val_data), 'test_examples': len(test_data),
    'test_loss': test_metrics['loss'], 'test_accuracy': test_metrics['accuracy'],
    'macro_precision': float(macro_precision), 'macro_recall': float(macro_recall), 'macro_f1': float(macro_f1),
    'test_mistakes': int(len(mistakes)), 'selected_train_accuracy': float(selected.train_accuracy),
    'selected_validation_accuracy': float(selected.val_accuracy), 'selected_accuracy_gap_pp': float(selected_gap * 100),
    'final_accuracy_gap_pp': float(final_gap * 100), 'training_seconds': elapsed_seconds,
    'dataset_sha256': archive_sha256,
    'versions': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__}
}
(artifact_dir / 'run-summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
print(json.dumps(run_summary, indent=2))
print('Verification passed. Training history and run summary exported.')
